In [ ]:
import numpy as np

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize OpenAI Client
client = OpenAI(api_key="")

# 2. Knowledge Base
documents = [
    {"id": "HR-001", "text": "Employees receive 26 annual leave days."},
    {"id": "HR-002", "text": "Leave requests must be approved by the manager."},
    {"id": "HR-003", "text": "Employees have supplementary health insurance."},
    {"id": "HR-004", "text": "Monthly salaries are paid at the end of each month."},
    {"id": "HR-005", "text": "Employees can work remotely two days per week."},
    {"id": "HR-006", "text": "Employees can request unpaid leave."},
]

# 3. Load Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 4. Generate Document Embeddings
document_texts = [doc["text"] for doc in documents]

document_embeddings = embedding_model.encode(
    document_texts, convert_to_numpy=True, normalize_embeddings=True
)


# 5. Retrieve Relevant Chunks
def retrieve_chunks(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append(
            {
                "id": documents[index]["id"],
                "text": documents[index]["text"],
                "score": float(scores[index]),
            }
        )

    return results


# 6. Construct Context
def construct_context(retrieved_chunks):

    return "\n\n".join(
        f"[Source: {chunk['id']}]\n{chunk['text']}" for chunk in retrieved_chunks
    )


# 7. Construct Prompt Messages
def construct_messages(query, context):

    system_instruction = """
You are an HR assistant.

Rules:
- Answer using only the provided context.
- Do not invent information.
- If the answer is unavailable in the context,
  clearly say that you don't know.
- Keep your answer clear and concise.
"""

    user_message = f"""
Context:
{context}

Question:
{query}
"""

    return [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message},
    ]


# 8. Send Prompt to LLM
def ask_llm(messages):

    response = client.chat.completions.create(model="gpt-4.1-mini", messages=messages)

    return response.choices[0].message.content


# 9. Execute RAG Pipeline
query = "How many annual leave days do employees receive?"

retrieved_chunks = retrieve_chunks(query, top_k=3)

context = construct_context(retrieved_chunks)

messages = construct_messages(query, context)

answer = ask_llm(messages)

# 10. Display Results
print("QUESTION:")
print(query)

print("\nCONTEXT:")
print(context)

print("\nLLM ANSWER:")
print(answer)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 982.85it/s]


RateLimitError: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.